# Polars Attributes: Examples of Every Attribute
Basics to advanced. Attributes are accessed **without parentheses** (`df.shape`), unlike methods (`df.head()`).

> **Polars vs pandas:** Polars has far fewer plain attributes. Many things that are properties in pandas are **methods** in Polars (`s.dt.year()`, `df.is_empty()`), and Polars has **no index**. Section 9 maps pandas attributes to their Polars equivalents.

**Sections**
1. Module-level attributes
2. `DataFrame` attributes
3. `Series` attributes
4. `LazyFrame` attributes
5. Data types and their attributes
6. Namespaces: `.str`, `.dt`, `.list`, `.arr`, `.struct`, `.cat`, `.bin`, `.name`, `.meta`, `.plot`, `.style`
7. Selectors (`polars.selectors`)
8. `pl.Config` options
9. pandas to Polars attribute map

In [ ]:
import warnings
from datetime import date, datetime

import polars as pl
import polars.selectors as cs

print("Polars version:", pl.__version__)


def show(label, fn):
    """Run fn() and print the result. A version-specific feature prints n/a instead of stopping."""
    try:
        print(f"{label:30}: {fn()}")
    except Exception as e:
        print(f"{label:30}: n/a ({type(e).__name__}: {e})")


df = pl.DataFrame({
    "name":   ["Asha", "Ravi", "Meena", "Karan", "Sita", "Dev"],
    "age":    [25, 32, 28, 45, 38, 29],
    "city":   ["Delhi", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi"],
    "salary": [50000.0, 72000.5, 61000.0, 95000.0, 83000.0, 58000.0],
    "joined": [date(2020, 1, 15), date(2018, 6, 1), date(2021, 3, 20),
               date(2015, 11, 30), date(2019, 8, 10), date(2022, 2, 28)],
    "skills": [["python", "sql"], ["r"], ["python"],
               ["sql", "excel", "bi"], ["python", "ml"], ["go"]],
    "active": [True, True, False, True, False, True],
})
df

## 1. Module-Level Attributes
| Item | Meaning |
|---|---|
| `pl.__version__` | installed Polars version |
| `pl.Config.state()` | options that are currently set |
| `pl.thread_pool_size()` | threads used by the engine |
| `pl.get_index_type()` | dtype used for row indexes (`UInt32` or `UInt64`) |
| `pl.selectors` | column selectors (section 7) |

In [ ]:
print("version:", pl.__version__)
show("Config.state()", lambda: pl.Config.state())
show("thread_pool_size()", lambda: pl.thread_pool_size())
show("get_index_type()", lambda: pl.get_index_type())
show("top-level public names", lambda: len([n for n in dir(pl) if not n.startswith("_")]))

## 2. `DataFrame` Attributes
| Attribute | Meaning |
|---|---|
| `df.shape` | (rows, columns) tuple |
| `df.height` | number of rows |
| `df.width` | number of columns |
| `df.columns` | list of column names |
| `df.dtypes` | list of data types |
| `df.schema` | ordered mapping `{name: dtype}` |
| `df.flags` | per-column flags such as sortedness |
| `df.plot` | plotting namespace (needs `altair`) |
| `df.style` | table styling (needs `great_tables`) |

In [ ]:
print("shape  :", df.shape)
print("height :", df.height)
print("width  :", df.width)
print("len(df):", len(df), "(same as height)")
print("columns:", df.columns)
print("dtypes :", df.dtypes)

In [ ]:
# schema: ordered mapping of column name -> dtype
print("schema      :", df.schema)
print("type        :", type(df.schema).__name__)
print("age dtype   :", df.schema["age"])
print("items       :", list(df.schema.items())[:3], "...")
show("schema.names()", lambda: df.schema.names())
show("schema.dtypes()", lambda: df.schema.dtypes())
show("schema.len()", lambda: df.schema.len())
show("schema.to_python()", lambda: df.schema.to_python())

# Build a schema explicitly
sch = pl.Schema({"id": pl.Int32, "label": pl.String})
print("explicit schema:", sch)

In [ ]:
# Empty DataFrame
empty = pl.DataFrame()
print("empty shape     :", empty.shape)
print("empty is_empty():", empty.is_empty(), "| df.is_empty():", df.is_empty())

# flags describe known metadata, such as sort order
sorted_df = df.sort("age")
show("df.flags (unsorted)", lambda: df.flags)
show("sorted_df.flags", lambda: sorted_df.flags)

In [ ]:
# Related inspection METHODS (parentheses needed)
print("estimated_size :", df.estimated_size("kb"), "KB")
print("n_chunks       :", df.n_chunks())
print("null_count     :", df.null_count().row(0))
print("is_unique      :", df["name"].is_unique().all())
df.describe()

## 3. `Series` Attributes
| Attribute | Meaning |
|---|---|
| `s.name` | name of the Series |
| `s.dtype` | data type |
| `s.shape` | `(length,)` |
| `s.flags` | sortedness and other flags |
| `s.str` / `s.dt` / `s.list` / `s.arr` / `s.struct` / `s.cat` / `s.bin` | namespaces |

In [ ]:
s = df["salary"]
print("name :", s.name)
print("dtype:", s.dtype)
print("shape:", s.shape)
print("len  :", len(s))
show("flags", lambda: s.flags)
show("flags after sort", lambda: s.sort().flags)

# Rename and cast
s2 = s.alias("pay")
print("renamed:", s2.name, "| original still:", s.name)
print("cast   :", s.cast(pl.Int32).dtype)

In [ ]:
# Series built directly
nums = pl.Series("nums", [3, 1, 2])
print(nums)
print("dtype    :", nums.dtype, "| is numeric:", nums.dtype.is_numeric())
print("empty    :", pl.Series([], dtype=pl.Int64).shape)
print("with null:", pl.Series("x", [1, None, 3]).null_count(), "null(s)")
print("all dtypes in df:", {c: str(df[c].dtype) for c in df.columns})

## 4. `LazyFrame` Attributes
`df.lazy()` builds a query plan. Reading `lf.columns`, `lf.dtypes`, `lf.schema` or `lf.width` forces Polars to resolve the schema, which can be expensive, so recent versions warn and recommend **`collect_schema()`**.

In [ ]:
lf = df.lazy().filter(pl.col("age") > 26).select("name", "age", "salary")

schema = lf.collect_schema()
print("collect_schema():", schema)
print("names           :", schema.names())
print("len             :", schema.len())

# The older attribute style still works, but may warn
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print("lf.columns :", lf.columns)
    print("lf.dtypes  :", lf.dtypes)
    print("lf.schema  :", lf.schema)
    print("lf.width   :", lf.width)

In [ ]:
# Peek at the plan, then run it
print(lf.explain())
lf.collect()

## 5. Data Types and Their Attributes
Every dtype is a class in `pl`. Parameterised types carry their own attributes.

| Type | Attributes |
|---|---|
| `pl.Datetime(time_unit, time_zone)` | `time_unit`, `time_zone` |
| `pl.Duration(time_unit)` | `time_unit` |
| `pl.List(inner)` | `inner` |
| `pl.Array(inner, shape)` | `inner`, `size`, `shape` |
| `pl.Struct(fields)` | `fields` |
| `pl.Decimal(precision, scale)` | `precision`, `scale` |
| `pl.Enum(categories)` | `categories` |

In [ ]:
# Built-in types
types = [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
         pl.Float32, pl.Float64, pl.Boolean, pl.String, pl.Date, pl.Time, pl.Binary, pl.Null]
for t in types:
    show(f"{str(t):10}", lambda t=t: f"integer={t.is_integer()} float={t.is_float()} "
                                       f"numeric={t.is_numeric()} temporal={t.is_temporal()}")

In [ ]:
# Parameterised types and their attributes
dt = pl.Datetime("us", "UTC")
print("Datetime        :", dt, "| time_unit:", dt.time_unit, "| time_zone:", dt.time_zone)

du = pl.Duration("ms")
print("Duration        :", du, "| time_unit:", du.time_unit)

li = pl.List(pl.Int64)
print("List            :", li, "| inner:", li.inner)

ar = pl.Array(pl.Int64, 3)
show("Array.inner", lambda: ar.inner)
show("Array.size", lambda: ar.size)
show("Array.shape", lambda: ar.shape)

st = pl.Struct({"x": pl.Int64, "y": pl.String})
print("Struct          :", st)
print("Struct fields   :", st.fields)
print("field names     :", [f.name for f in st.fields], "| dtypes:", [f.dtype for f in st.fields])

de = pl.Decimal(10, 2)
show("Decimal.precision", lambda: de.precision)
show("Decimal.scale", lambda: de.scale)

en = pl.Enum(["low", "mid", "high"])
show("Enum.categories", lambda: en.categories.to_list())

In [ ]:
# Comparing and inspecting dtypes
print("List(Int64) == pl.List        :", pl.List(pl.Int64) == pl.List)
print("List(Int64) == List(String)   :", pl.List(pl.Int64) == pl.List(pl.String))
print("Datetime('us') == pl.Datetime :", pl.Datetime("us") == pl.Datetime)

show("List.base_type()", lambda: pl.List(pl.Int64).base_type())
show("Int64.is_(pl.Int64)", lambda: pl.Int64.is_(pl.Int64))
show("List(Int64).is_nested()", lambda: pl.List(pl.Int64).is_nested())
show("Int64 is_numeric()", lambda: pl.Int64.is_numeric())

print("\nColumn dtypes in df:")
for name, dtype in df.schema.items():
    print(f"  {name:8} {str(dtype):16} nested={dtype.is_nested()}")

## 6. Namespaces
Namespaces group specialised tools. In Polars they hold **methods** (`.str.to_uppercase()`), unlike pandas where many are properties.

| Namespace | Works on |
|---|---|
| `.str` | String |
| `.dt` | Date, Datetime, Time, Duration |
| `.list` | List |
| `.arr` | Array (fixed-size) |
| `.struct` | Struct |
| `.cat` | Categorical |
| `.bin` | Binary |
| `.name` | column names of an expression |
| `.meta` | introspect an expression |
| `.plot` / `.style` | quick charts / table styling |

### 6.1 `.str`

In [ ]:
df.select(
    "name",
    upper=pl.col("name").str.to_uppercase(),
    lower=pl.col("name").str.to_lowercase(),
    n_chars=pl.col("name").str.len_chars(),
    n_bytes=pl.col("name").str.len_bytes(),
    has_a=pl.col("name").str.contains("a"),
    starts_R=pl.col("name").str.starts_with("R"),
    ends_a=pl.col("name").str.ends_with("a"),
    first2=pl.col("name").str.slice(0, 2),
    swapped=pl.col("name").str.replace("a", "@"),
    swapped_all=pl.col("name").str.replace_all("a", "@"),
    padded=pl.col("name").str.pad_start(8, "*"),
)

In [ ]:
text = pl.DataFrame({"raw": ["  hello world ", "a-b-c", "ID: 42"]})
text.select(
    "raw",
    stripped=pl.col("raw").str.strip_chars(),
    split=pl.col("raw").str.split("-"),
    digits=pl.col("raw").str.extract(r"(\d+)", 1),
)

### 6.2 `.dt` (dates and times)

In [ ]:
df.select(
    "joined",
    year=pl.col("joined").dt.year(),
    month=pl.col("joined").dt.month(),
    day=pl.col("joined").dt.day(),
    weekday=pl.col("joined").dt.weekday(),          # Monday = 1
    ordinal_day=pl.col("joined").dt.ordinal_day(),
    week=pl.col("joined").dt.week(),
    quarter=pl.col("joined").dt.quarter(),
    leap=pl.col("joined").dt.is_leap_year(),
    month_start=pl.col("joined").dt.month_start(),
    month_end=pl.col("joined").dt.month_end(),
    text=pl.col("joined").dt.strftime("%d %b %Y"),
)

In [ ]:
# Datetime columns: time parts, truncation, epoch, time zones
ts = pl.DataFrame({"ts": pl.datetime_range(datetime(2024, 3, 1, 0), datetime(2024, 3, 1, 18),
                                            "6h", eager=True)})
ts.select(
    "ts",
    hour=pl.col("ts").dt.hour(),
    minute=pl.col("ts").dt.minute(),
    second=pl.col("ts").dt.second(),
    date=pl.col("ts").dt.date(),
    time=pl.col("ts").dt.time(),
    epoch_s=pl.col("ts").dt.epoch("s"),
    floor_12h=pl.col("ts").dt.truncate("12h"),
    utc=pl.col("ts").dt.replace_time_zone("UTC"),
    india=pl.col("ts").dt.replace_time_zone("UTC").dt.convert_time_zone("Asia/Kolkata"),
)

In [ ]:
# Durations: difference between two dates
dur = df.select(
    "name",
    days_since_joining=(pl.lit(date(2025, 1, 1)) - pl.col("joined")).dt.total_days(),
    hours=(pl.lit(date(2025, 1, 1)) - pl.col("joined")).dt.total_hours(),
)
print(dur.schema)
dur

### 6.3 `.list`

In [ ]:
df.select(
    "name",
    "skills",
    n=pl.col("skills").list.len(),
    first=pl.col("skills").list.first(),
    last=pl.col("skills").list.last(),
    item0=pl.col("skills").list.get(0),
    knows_python=pl.col("skills").list.contains("python"),
    joined=pl.col("skills").list.join(", "),
    sorted=pl.col("skills").list.sort(),
    n_unique=pl.col("skills").list.n_unique(),
)

In [ ]:
# explode turns each list item into its own row
df.select("name", "skills").explode("skills")

### 6.4 `.arr` (fixed-size arrays)

In [ ]:
arrs = pl.DataFrame({"v": pl.Series([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=pl.Array(pl.Int64, 3))})
print("dtype :", arrs.schema["v"])
arrs.select(
    "v",
    total=pl.col("v").arr.sum(),
    smallest=pl.col("v").arr.min(),
    largest=pl.col("v").arr.max(),
    first=pl.col("v").arr.get(0),
    has5=pl.col("v").arr.contains(5),
)

### 6.5 `.struct`

In [ ]:
sdf = pl.DataFrame({"point": [{"x": 1, "y": 2}, {"x": 3, "y": 4}, {"x": 5, "y": 6}]})
print("dtype :", sdf.schema["point"])
show("struct.fields", lambda: sdf["point"].struct.fields)
show("struct.schema", lambda: sdf["point"].struct.schema)

sdf.select(
    "point",
    x=pl.col("point").struct.field("x"),
    y=pl.col("point").struct.field("y"),
)

In [ ]:
# unnest spreads struct fields into separate columns
sdf.unnest("point")

### 6.6 `.cat` and `Enum`

In [ ]:
cat = pl.Series("level", ["low", "high", "low", "mid"], dtype=pl.Categorical)
print("dtype     :", cat.dtype)
show("cat.get_categories()", lambda: cat.cat.get_categories().to_list())
show("physical codes", lambda: cat.to_physical().to_list())

enum_dtype = pl.Enum(["low", "mid", "high"])
lev = pl.Series("level", ["low", "high", "mid"], dtype=enum_dtype)
print("Enum dtype:", lev.dtype)
show("Enum categories", lambda: lev.dtype.categories.to_list())
show("ordered compare (low < high)", lambda: (lev < "high").to_list())

### 6.7 `.bin`

In [ ]:
bs = pl.Series("data", [b"abc", b"hello", b"xyz"])
print("dtype:", bs.dtype)
show("bin.size()", lambda: bs.bin.size().to_list())
show("bin.contains(b'l')", lambda: bs.bin.contains(b"l").to_list())
show("bin.starts_with(b'a')", lambda: bs.bin.starts_with(b"a").to_list())
show("bin.encode('hex')", lambda: bs.bin.encode("hex").to_list())

### 6.8 `.name` (rename columns inside expressions)

In [ ]:
df.select(pl.col("age", "salary").name.suffix("_new"))

In [ ]:
print(df.select(pl.col("age", "salary").name.prefix("emp_")).columns)
print(df.select(pl.col("age", "salary").name.to_uppercase()).columns)
print(df.select(pl.col("age", "salary").name.to_lowercase()).columns)
print(df.select(pl.col("age").alias("renamed").name.keep()).columns, "(keep restores the root name)")

### 6.9 `.meta` (introspect expressions)

In [ ]:
e = (pl.col("age") + pl.col("salary")).alias("total")
show("meta.output_name()", lambda: e.meta.output_name())
show("meta.root_names()", lambda: e.meta.root_names())
show("meta.undo_aliases()", lambda: e.meta.undo_aliases())
show("meta.has_multiple_outputs()", lambda: e.meta.has_multiple_outputs())
show("meta.is_column()", lambda: pl.col("age").meta.is_column())
show("meta.is_regex_projection()", lambda: pl.col("^a.*$").meta.is_regex_projection())
show("str(expression)", lambda: str(e))

### 6.10 `.plot` and `.style`
Both need optional packages (`altair` for plots, `great_tables` for styling).

In [ ]:
show("type(df.plot)", lambda: type(df.plot).__name__)
show("type(df.style)", lambda: type(df.style).__name__)

try:
    chart = df.plot.bar(x="name", y="salary")
    print("chart type:", type(chart).__name__)
    chart
except Exception as e:
    print("plot needs altair:", type(e).__name__)

## 7. Selectors (`polars.selectors`, imported as `cs`)
Pick columns by type, name or pattern. Selectors can be combined with `|`, `&`, `-` and `~`.

| Selector | Picks |
|---|---|
| `cs.numeric()` / `cs.integer()` / `cs.float()` | numeric columns |
| `cs.string()` / `cs.boolean()` / `cs.temporal()` | by type |
| `cs.by_dtype(...)` / `cs.by_name(...)` | explicit |
| `cs.starts_with()` / `cs.ends_with()` / `cs.contains()` / `cs.matches()` | by name pattern |
| `cs.first()` / `cs.last()` / `cs.all()` | by position |

In [ ]:
def names(sel):
    return list(cs.expand_selector(df, sel))

print("numeric     :", names(cs.numeric()))
print("integer     :", names(cs.integer()))
print("float       :", names(cs.float()))
print("string      :", names(cs.string()))
print("boolean     :", names(cs.boolean()))
print("temporal    :", names(cs.temporal()))
print("by_dtype    :", names(cs.by_dtype(pl.Float64, pl.Date)))
print("by_name     :", names(cs.by_name("name", "city")))
print("starts_with :", names(cs.starts_with("s")))
print("ends_with   :", names(cs.ends_with("e")))
print("contains    :", names(cs.contains("i")))
print("matches     :", names(cs.matches("^[a-c]")))
print("first / last:", names(cs.first()), names(cs.last()))
print("all         :", names(cs.all()))

In [ ]:
# Combine selectors
print("numeric minus age     :", names(cs.numeric() - cs.by_name("age")))
print("string OR boolean     :", names(cs.string() | cs.boolean()))
print("numeric AND ends 'e'  :", names(cs.numeric() & cs.ends_with("e")))
print("NOT numeric           :", names(~cs.numeric()))
print("exclude name, skills  :", names(cs.exclude("name", "skills")))

# Use in expressions
df.select(cs.numeric()).with_columns(cs.float().round(0).name.suffix("_rounded"))

## 8. `pl.Config` Options
Controls how DataFrames are printed. Use it as a **context manager** to change options temporarily.

| Option | Meaning |
|---|---|
| `tbl_rows` | rows shown |
| `tbl_cols` | columns shown |
| `fmt_str_lengths` | max characters per string |
| `tbl_formatting` | table border style |
| `tbl_hide_column_data_types` | hide dtype row |
| `tbl_hide_dataframe_shape` | hide the shape line |
| `float_precision` | decimals for floats |

In [ ]:
with pl.Config(tbl_rows=3, tbl_cols=4):
    print(df)

In [ ]:
with pl.Config(tbl_formatting="MARKDOWN", tbl_hide_column_data_types=True,
               tbl_hide_dataframe_shape=True, float_precision=1):
    print(df.select("name", "age", "salary").head(3))

In [ ]:
# Global setters (stay on until restored)
pl.Config.set_tbl_rows(2)
show("state after set_tbl_rows", lambda: pl.Config.state())
pl.Config.restore_defaults()
show("state after restore_defaults", lambda: pl.Config.state())

## 9. pandas to Polars Attribute Map
| pandas | Polars | Note |
|---|---|---|
| `df.shape` | `df.shape` | same |
| `df.columns` | `df.columns` | list, not an Index |
| `df.dtypes` | `df.dtypes` / `df.schema` | list / mapping |
| `df.size` | `df.height * df.width` | no `size` attribute |
| `df.ndim` | always 2 | no attribute |
| `df.index` | none | Polars has no index; use `with_row_index()` |
| `df.values` | `df.to_numpy()` | method |
| `df.empty` | `df.is_empty()` | method |
| `df.T` | `df.transpose()` | method |
| `df.loc[]` / `df.iloc[]` | `df[...]`, `filter()`, `row()` | no label index |
| `df.at[]` / `df.iat[]` | `df.item(row, col)` | method |
| `s.name` | `s.name` | same |
| `s.dt.year` | `s.dt.year()` | method |
| `s.str.upper()` | `s.str.to_uppercase()` | different name |
| `s.cat.categories` | `s.cat.get_categories()` | method |
| `df.attrs` | none | no metadata dict |

In [ ]:
# size and ndim equivalents
print("size equivalent:", df.height * df.width)
print("ndim           : 2 (always)")

# values, T, empty
print("to_numpy shape :", df.select(cs.numeric()).to_numpy().shape)
df.select("name", "age", "salary").transpose(include_header=True, header_name="field")

In [ ]:
# loc / iloc / at / iat equivalents
print("row 0 (tuple)     :", df.row(0))
print("row 0 (dict)      :", df.row(0, named=True))
print("item(0, 'name')   :", df.item(0, "name"))
print("df[0, 'name']     :", df[0, "name"])
print("column as Series  :", df["name"].to_list())
print("rows 1..2         :")
print(df[1:3].select("name", "age"))
print("filter (like loc) :")
print(df.filter(pl.col("age") > 30).select("name", "age"))

In [ ]:
# No index: add one explicitly when you need row numbers
df.with_row_index("idx").select("idx", "name", "age").head(3)

In [ ]:
# Convert to and from pandas (needs pandas and pyarrow)
try:
    pdf = df.select("name", "age").to_pandas()
    print("to_pandas   :", type(pdf).__name__, pdf.shape)
    print("from_pandas :", pl.from_pandas(pdf).shape)
except Exception as e:
    print("needs pandas + pyarrow:", type(e).__name__)

## Summary
- **DataFrame:** `shape`, `height`, `width`, `columns`, `dtypes`, `schema`, `flags`, `plot`, `style`
- **Series:** `name`, `dtype`, `shape`, `flags`, plus namespaces
- **LazyFrame:** prefer `collect_schema()` over `columns` / `dtypes` / `schema` / `width`
- **dtype attributes:** `time_unit`, `time_zone`, `inner`, `size`, `fields`, `precision`, `scale`, `categories`
- **Namespaces:** `.str`, `.dt`, `.list`, `.arr`, `.struct`, `.cat`, `.bin`, `.name`, `.meta`
- **Selectors:** `cs.numeric()`, `cs.string()`, `cs.by_dtype()`, `cs.starts_with()`, set operations
- **Config:** `pl.Config(...)` context manager, `state()`, `restore_defaults()`
- **No index** in Polars: use `with_row_index()`